## 1. 路径配置与标签加载

In [ ]:
## 1. 路径配置与标签加载
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.patches import Rectangle, Patch
import h5py
import pandas as pd
from collections import Counter
from scipy.stats import entropy

# ============================================================================
# 路径配置 - 修改此处切换被试
# ============================================================================

SUBJECT_ID = "ODP_01_qhlazec"
SPLIT_TYPE = "test"

BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")

if SPLIT_TYPE == "test":
    fold_candidates = list(BASE_RESULT_DIR.glob(f"*_test_{SUBJECT_ID}"))
    FOLD_DIR = fold_candidates[0] if len(fold_candidates) > 0 else BASE_RESULT_DIR / f"fold_01_test_{SUBJECT_ID}"
else:
    FOLD_DIR = BASE_RESULT_DIR / "fold_01_test_ODP_01_qhlazec"

DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")
DOWNSAMPLED_FILE = DATA_ROOT / f"{SUBJECT_ID}_downsampled.npz"
PRED_SOFTMAX_FILE = FOLD_DIR / "pred_3d" / f"{SPLIT_TYPE}_{SUBJECT_ID}_pred_softmax_3d.npz"
GT_3D_FILE = DATA_ROOT / "3d" / f"{SUBJECT_ID}_3d.npz"
LABEL_EXCEL = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")

OUTPUT_DIR = FOLD_DIR / "figs" / "confidence_overlay"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHANNEL_MPRAGE = 341
CHANNEL_QSM = 350

# ============================================================================
# 可视化参数
# ============================================================================

TAU = 0.4
N_CLASSES = 102
SLICE_AXIS = 'axial'
AUTO_SELECT_SLICES = True
MANUAL_SLICES = [30, 50, 70]
CONTOUR_LEVELS = [0.3]
CONTOUR_LINEWIDTH = 1.5

ZOOM_REGIONS = {
    "cortex_white": {"z": None, "x": None, "y": None, "hw": 20},
    "basal_ganglia": {"z": None, "x": None, "y": None, "hw": 20},
    "brainstem": {"z": None, "x": None, "y": None, "hw": 20}
}

print("✓ 路径配置完成")
print(f"  被试: {SUBJECT_ID}, 类型: {SPLIT_TYPE}")
print(f"  Fold: {FOLD_DIR.name}")
print(f"  输出: {OUTPUT_DIR}")

# ============================================================================
# 加载FreeSurfer标签映射
# ============================================================================

print("\n加载FreeSurfer标签映射...")
label_df = pd.read_excel(LABEL_EXCEL)
valid_labels = label_df[label_df['one_hot_loc_alex_label'] != '[]'].copy()
valid_labels['label_idx'] = valid_labels['one_hot_loc_alex_label'].astype(int)

LABEL_NAMES = {0: 'Background'}
LABEL_COLORS_RGB = {0: (0, 0, 0)}

for idx, row in valid_labels.iterrows():
    label_idx = int(row['label_idx'])
    if 1 <= label_idx <= N_CLASSES and label_idx not in LABEL_NAMES:
        LABEL_NAMES[label_idx] = row['tissue_name'].strip("'")
        LABEL_COLORS_RGB[label_idx] = (row['R']/255, row['G']/255, row['B']/255)

colors_list = [LABEL_COLORS_RGB.get(i, (0, 0, 0)) for i in range(N_CLASSES)]
cmap_freesurfer = mcolors.ListedColormap(colors_list)

print(f"✓ 加载了 {len(LABEL_NAMES)} 个标签")

In [ ]:
## 2. 加载数据并计算置信度与不确定性
# print("加载数据...")

# 预测概率
pred_data = np.load(PRED_SOFTMAX_FILE)
pred_softmax = pred_data['pred_softmax_3d']
print(f"  预测概率: {pred_softmax.shape}")

# Ground Truth
gt_data = np.load(GT_3D_FILE)
gt_proba = gt_data['proba_labels']
region_mask = gt_data['region_mask_lr']
print(f"  Ground Truth: {gt_proba.shape}")

# MPRAGE底图
if DOWNSAMPLED_FILE.exists():
    downsampled_data = np.load(DOWNSAMPLED_FILE)
    data_lr = downsampled_data['data_lr']
    anatomy_img = data_lr[..., CHANNEL_MPRAGE]
    anatomy_name = "MPRAGE"
    print(f"  MPRAGE底图: {anatomy_img.shape}")
else:
    anatomy_img = np.zeros(pred_softmax.shape[:3])
    anatomy_name = "No Anatomy"

# ============================================================================
# 计算置信度
# ============================================================================

print("\n计算置信度与不确定性...")

top2_indices = np.argsort(pred_softmax, axis=-1)[..., -2:]
top1_class = top2_indices[..., 1]
top2_class = top2_indices[..., 0]

top2_probs = np.take_along_axis(pred_softmax, top2_indices, axis=-1)
p1 = top2_probs[..., 1]
p2 = top2_probs[..., 0]

# 边际差 (margin)
margin = p1 - p2  # (Z, X, Y)
alpha_map = np.clip(margin / TAU, 0, 1)
alpha_map[region_mask == 0] = 0

# ============================================================================
# 计算熵图（不确定性）
# ============================================================================

# H(p) = -Σ_k p_k log(p_k)
# 使用scipy.stats.entropy，axis=-1计算每个体素的熵
epsilon = 1e-10  # 避免log(0)
pred_softmax_safe = np.clip(pred_softmax, epsilon, 1.0)

# entropy使用自然对数，转换为bits需要除以log(2)
entropy_map = entropy(pred_softmax_safe.T, axis=0).T  # scipy要求axis=0，所以转置
entropy_map = entropy_map / np.log(2)  # 转为bits

# 应用ROI掩码
entropy_map_masked = entropy_map.copy()
entropy_map_masked[region_mask == 0] = 0

# 统计
entropy_roi = entropy_map[region_mask > 0]
margin_roi = margin[region_mask > 0]

print(f"  边际差 (Margin, ROI内):")
print(f"    Mean={margin_roi.mean():.4f}, Median={np.median(margin_roi):.4f}")
print(f"    Range=[{margin_roi.min():.4f}, {margin_roi.max():.4f}]")

print(f"  熵 (Entropy, ROI内):")
print(f"    Mean={entropy_roi.mean():.4f} bits, Median={np.median(entropy_roi):.4f} bits")
print(f"    Range=[{entropy_roi.min():.4f}, {entropy_roi.max():.4f}] bits")
print(f"    Max possible: {np.log2(N_CLASSES):.2f} bits (uniform distribution)")

# ============================================================================
# 自动选择切片
# ============================================================================

if AUTO_SELECT_SLICES:
    print("\n自动选择切片...")
    axis_size = pred_softmax.shape[0]
    slice_scores = []
    
    for z in range(axis_size):
        mask_slice = region_mask[z] > 0
        if mask_slice.sum() > 100:
            # 使用熵的标准差作为信息量度量
            entropy_std = entropy_map[z][mask_slice].std()
            mask_ratio = mask_slice.sum() / mask_slice.size
            slice_scores.append((z, entropy_std * mask_ratio))
    
    slice_scores.sort(key=lambda x: x[1], reverse=True)
    selected_slices = []
    for z, score in slice_scores:
        if len(selected_slices) == 0 or all(abs(z - s) >= 10 for s in selected_slices):
            selected_slices.append(z)
        if len(selected_slices) >= 3:
            break
    
    MANUAL_SLICES = sorted(selected_slices)
    print(f"  选择的切片: {MANUAL_SLICES}")

print("\n✓ 数据准备完成")

In [ ]:
# ====================== 绘制三联图（换行与不重叠版） ======================
import textwrap

# 建议：关闭 constrained_layout，手动控制边距，给右侧图例留白
fig, axes = plt.subplots(1, 3, figsize=(36, 12))
fig.patch.set_facecolor('black')

# —— 工具：包一层自动换行 —— 
def wrap(s, width=28):
    # 你可以按需要调宽度（越小越容易换行）
    return textwrap.fill(s, width=width)

def set_title(ax, s, fontsize=14, pad=8):
    txt = ax.set_title(wrap(s), fontsize=fontsize, fontweight='bold',
                       color='white', pad=pad, wrap=True)
    # 增加行距，避免多行贴在一起
    txt.set_linespacing(1.15)

def plot_categorical_map(ax, label_map, mask, title, show_red_mask=False, red_mask_data=None):
    label_rgb = np.zeros((*label_map.shape, 3))
    for class_idx in range(N_CLASSES):
        class_mask = (label_map == class_idx) & (mask > 0)
        if class_mask.any():
            color = LABEL_COLORS_RGB.get(class_idx, (0, 0, 0))
            label_rgb[class_mask] = color

    ax.imshow(label_rgb, aspect='auto', interpolation='nearest')

    if show_red_mask and red_mask_data is not None:
        red_overlay = np.zeros((*label_map.shape, 4))
        red_overlay[red_mask_data] = RED_MASK_COLOR
        ax.imshow(red_overlay, aspect='auto', interpolation='nearest')

    ax.contour(mask, levels=[0.5], colors='cyan',
               linewidths=1.0, linestyles='--', alpha=0.5)
    ax.axis('off')
    set_title(ax, title, fontsize=14, pad=10)

# (a) Reference Mode
plot_categorical_map(
    axes[0],
    ref_mode,
    mask_slice,
    f'(a) Reference Mode\n(argmax of soft reference)\nSlice {SLICE_IDX}'
)

# (b) Top-1 Prediction
plot_categorical_map(
    axes[1],
    pred_top1,
    mask_slice,
    f'(b) Top-1 Prediction\n(argmax of p_pred, post-T)\nSlice {SLICE_IDX}'
)

# (c) Top-3 Union + 红遮罩
plot_categorical_map(
    axes[2],
    ref_mode,  # 用参考颜色做底色
    mask_slice,
    f'(c) Top-{K_TOP} Union + Uncovered Mask\n'
    f'Red = Ref mode NOT in pred Top-{K_TOP}\n'
    f'Coverage: {100-red_ratio:.1f}%, Uncovered: {red_ratio:.1f}%',
    show_red_mask=True,
    red_mask_data=red_mask
)

# —— 图例：放右侧，避免挤占子图 —— 
legend_labels = []
for class_idx in [0, 2, 7, 10, 41, 42]:
    if class_idx in LABEL_NAMES:
        legend_labels.append(
            Patch(facecolor=LABEL_COLORS_RGB[class_idx],
                  edgecolor='white', label=LABEL_NAMES[class_idx])
        )
legend_labels.append(
    Patch(facecolor=RED_MASK_COLOR[:3], alpha=RED_MASK_COLOR[3],
          edgecolor='white', label='Uncovered (Red Mask)')
)

fig.legend(handles=legend_labels, loc='center left', bbox_to_anchor=(0.985, 0.5),
           frameon=True, facecolor='black', edgecolor='white',
           fontsize=10, labelcolor='white')

# —— 主标题：先手动换行，再留出顶端空间 —— 
suptitle_str = (
    f'Figure 1: Reference Mode | Top-1 Prediction | Top-{K_TOP} Union with Uncovered Regions\n'
    f'Subject: {SUBJECT_ID}, Slice: {SLICE_IDX}, {SPLIT_TYPE.upper()} set'
)
fig.suptitle(textwrap.fill(suptitle_str, width=70),
             fontsize=16, fontweight='bold', color='white', y=0.98)

# —— 关键：留白，防止重叠（右侧图例+上方主标题） —— 
# right 要小于 1 给 legend，top 要小于 1 给 suptitle，wspace 调小减少子图间空隙
fig.subplots_adjust(left=0.03, right=0.93, top=0.90, bottom=0.03, wspace=0.03)

plt.show()


In [ ]:
## Figure 1: 三联图 - Reference Mode | Top-1 | Top-3 Union + 未覆盖红遮罩
# ============================================================================
# ISMRM 图像制图工程师 - Figure 1 生成
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# ============================================================================
# 参数配置
# ============================================================================

SLICE_IDX = 17  # 可手动切换切片
TAU_HARD = True  # 使用硬规则（参考众数是否落入Top-3）
RED_MASK_COLOR = (220/255, 0, 0, 0.6)  # RGBA for red mask
K_TOP = 3  # Top-K

print(f"=== Figure 1 三联图生成 ===")
print(f"被试: {SUBJECT_ID}, 切片: {SLICE_IDX}")
print(f"规则: 硬规则（参考众数是否落入Top-{K_TOP}）")

# ============================================================================
# 提取切片数据
# ============================================================================

# 预测后验 (post-T)
p_pred_slice = pred_softmax[SLICE_IDX, :, :, :]  # (H, W, C=102)

# 软参考标签
p_ref_slice = gt_proba[SLICE_IDX, :, :, :]  # (H, W, C=102)

# ROI掩膜
mask_slice = region_mask[SLICE_IDX, :, :]  # (H, W)

# 解剖底图（可选）
anatomy_slice = anatomy_img[SLICE_IDX, :, :]

# ============================================================================
# 计算三个面板的内容
# ============================================================================

# (a) 参考众数（Reference mode）: argmax of soft reference
ref_mode = np.argmax(p_ref_slice, axis=-1)  # (H, W)

# (b) Top-1 预测
pred_top1 = np.argmax(p_pred_slice, axis=-1)  # (H, W)

# (c) Top-3 union + 红遮罩
# 获取 Top-3 classes (每个像素)
top3_indices = np.argsort(p_pred_slice, axis=-1)[..., -K_TOP:]  # (H, W, 3)

# 硬规则：检查 ref_mode 是否在 pred_top3 中
covered_mask = np.zeros(ref_mode.shape, dtype=bool)  # (H, W)
for i in range(ref_mode.shape[0]):
    for j in range(ref_mode.shape[1]):
        if mask_slice[i, j] > 0:  # 只在ROI内计算
            ref_class = ref_mode[i, j]
            pred_top3_classes = top3_indices[i, j, :]
            covered_mask[i, j] = ref_class in pred_top3_classes

# 红遮罩：ref_mode 不在 Top-3 中
red_mask = (~covered_mask) & (mask_slice > 0)  # (H, W)

# ============================================================================
# 统计信息（校验输出）
# ============================================================================

roi_pixels = mask_slice > 0
total_roi = roi_pixels.sum()
red_pixels = red_mask.sum()
red_ratio = 100 * red_pixels / total_roi if total_roi > 0 else 0

print(f"\n=== 统计信息 ===")
print(f"ROI 总像素数: {total_roi}")
print(f"红遮罩像素数: {red_pixels}")
print(f"红遮罩比例: {red_ratio:.2f}%")
print(f"Top-{K_TOP} 覆盖率: {100 - red_ratio:.2f}%")

# 软覆盖分数分布（额外信息，用于对比）
soft_coverage = np.zeros(ref_mode.shape)
for i in range(ref_mode.shape[0]):
    for j in range(ref_mode.shape[1]):
        if mask_slice[i, j] > 0:
            pred_top3_classes = top3_indices[i, j, :]
            soft_coverage[i, j] = p_ref_slice[i, j, pred_top3_classes].sum()

soft_cov_roi = soft_coverage[roi_pixels]
print(f"\n软覆盖分数（额外信息）:")
print(f"  Mean={soft_cov_roi.mean():.4f}, Median={np.median(soft_cov_roi):.4f}")
print(f"  Range=[{soft_cov_roi.min():.4f}, {soft_cov_roi.max():.4f}]")

# ============================================================================
# 绘制三联图
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(36, 12))

# 通用设置函数
def plot_categorical_map(ax, label_map, mask, title, show_red_mask=False, red_mask_data=None):
    """
    绘制分类图（统一LUT，无透明度）
    
    Parameters:
    - ax: matplotlib axis
    - label_map: (H, W) 整数标签
    - mask: (H, W) ROI掩膜
    - title: 标题
    - show_red_mask: 是否叠加红遮罩
    - red_mask_data: (H, W) 红遮罩布尔数组
    """
    # 背景黑色
    label_rgb = np.zeros((*label_map.shape, 3))
    
    # 应用FreeSurfer LUT
    for class_idx in range(N_CLASSES):
        class_mask = (label_map == class_idx) & (mask > 0)
        if class_mask.any():
            color = LABEL_COLORS_RGB.get(class_idx, (0, 0, 0))
            label_rgb[class_mask] = color
    
    # 显示分类图（无透明度）
    ax.imshow(label_rgb, aspect='auto', interpolation='nearest')
    
    # 叠加红遮罩（如果需要）
    if show_red_mask and red_mask_data is not None:
        red_overlay = np.zeros((*label_map.shape, 4))
        red_overlay[red_mask_data] = RED_MASK_COLOR
        ax.imshow(red_overlay, aspect='auto', interpolation='nearest')
    
    # ROI边界（青色虚线）
    ax.contour(mask, levels=[0.5], colors='cyan', 
               linewidths=1.0, linestyles='--', alpha=0.5)
    
    ax.set_title(title, fontsize=14, fontweight='bold', color='white', pad=10)
    ax.axis('off')

# ========================================================================
# (a) Reference Mode
# ========================================================================
plot_categorical_map(
    axes[0], 
    ref_mode, 
    mask_slice,
    f'(a) Reference Mode\n(argmax of soft reference)\nSlice {SLICE_IDX}'
)

# ========================================================================
# (b) Top-1 Prediction
# ========================================================================
plot_categorical_map(
    axes[1], 
    pred_top1, 
    mask_slice,
    f'(b) Top-1 Prediction\n(argmax of p_pred, post-T)\nSlice {SLICE_IDX}'
)

# ========================================================================
# (c) Top-3 Union + 红遮罩
# ========================================================================
plot_categorical_map(
    axes[2], 
    ref_mode,  # 显示参考颜色作为底色
    mask_slice,
    f'(c) Top-{K_TOP} Union + Uncovered Mask\n' +
    f'Red = Ref mode NOT in pred Top-{K_TOP}\n' +
    f'Coverage: {100-red_ratio:.1f}%, Uncovered: {red_ratio:.1f}%',
    show_red_mask=True,
    red_mask_data=red_mask
)

# ========================================================================
# 图例（可选，显示几个主要脑区颜色）
# ========================================================================

# 选择几个代表性标签
legend_labels = []
for class_idx in [0, 2, 7, 10, 41, 42]:  # 示例：背景+几个主要脑区
    if class_idx in LABEL_NAMES:
        legend_labels.append(
            Patch(facecolor=LABEL_COLORS_RGB[class_idx], 
                  edgecolor='white', label=LABEL_NAMES[class_idx])
        )

# 红遮罩图例
legend_labels.append(
    Patch(facecolor=RED_MASK_COLOR[:3], alpha=RED_MASK_COLOR[3],
          edgecolor='white', label='Uncovered (Red Mask)')
)

# 添加图例到右侧
fig.legend(handles=legend_labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
           frameon=True, facecolor='black', edgecolor='white', 
           fontsize=10, labelcolor='white')

# ========================================================================
# 统一设置
# ========================================================================

plt.tight_layout()
fig.patch.set_facecolor('black')

# 添加总标题
fig.suptitle(f'Figure 1: Reference Mode | Top-1 Prediction | Top-{K_TOP} Union with Uncovered Regions\n' +
             f'Subject: {SUBJECT_ID}, Slice: {SLICE_IDX}, {SPLIT_TYPE.upper()} set',
             fontsize=16, fontweight='bold', color='white', y=0.98)

plt.show()

# ============================================================================
# 保存统计数据（CSV）- 可选
# ============================================================================

# 软覆盖直方图（10 bins）
hist, bin_edges = np.histogram(soft_cov_roi, bins=10, range=(0, 1))
hist_data = {
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': hist,
    'percentage': 100 * hist / total_roi
}

print(f"\n=== 软覆盖分数直方图 (10 bins) ===")
for i in range(10):
    print(f"  [{hist_data['bin_left'][i]:.2f}, {hist_data['bin_right'][i]:.2f}): " +
          f"{hist_data['count'][i]} pixels ({hist_data['percentage'][i]:.2f}%)")

print(f"\n✓ Figure 1 生成完成！")

In [ ]:
# ====================== 四联图：无总标题，面板内标题 + 右侧全局色条 ======================
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from matplotlib.patches import Patch

# ---------- 配置：熵单位 ----------
# 选 "bits" 或 "nats"
ENTROPY_UNIT = "bits"

# ---------- 工具函数 ----------
def panel_title(ax, s):
    s = textwrap.fill(s, width=24)
    ax.text(
        0.01, 0.99, s,
        transform=ax.transAxes, va='top', ha='left',
        fontsize=13, fontweight='bold', color='white',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=(0,0,0,0.55), edgecolor='none')
    )

def plot_categorical_map(ax, label_map, mask, title, show_red_mask=False, red_mask_data=None):
    label_rgb = np.zeros((*label_map.shape, 3))
    for class_idx in range(N_CLASSES):
        class_mask = (label_map == class_idx) & (mask > 0)
        if class_mask.any():
            label_rgb[class_mask] = LABEL_COLORS_RGB.get(class_idx, (0, 0, 0))
    ax.imshow(label_rgb, aspect='auto', interpolation='nearest')

    if show_red_mask and red_mask_data is not None:
        red_overlay = np.zeros((*label_map.shape, 4))
        red_overlay[red_mask_data] = RED_MASK_COLOR
        ax.imshow(red_overlay, aspect='auto', interpolation='nearest')

    ax.contour(mask, levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--', alpha=0.5)
    ax.axis('off')
    panel_title(ax, title)

# ---------- 画布 ----------
fig, axes = plt.subplots(1, 4, figsize=(46, 12), constrained_layout=False)
fig.patch.set_facecolor('black')

# ---------- (a) Groundtruth ----------
plot_categorical_map(
    axes[0], ref_mode, mask_slice,
    '(a) Subject-specific parcellation (Groundtruth)'
)

# ---------- (b) Top-1 ----------
plot_categorical_map(
    axes[1], pred_top1, mask_slice,
    '(b) Top-1 prediction (post-T)'
)

# ---------- (c) Top-3 ∪ + 红遮罩 ----------
coverage = 100.0 - float(red_ratio)
missed = float(red_ratio)
plot_categorical_map(
    axes[2], ref_mode, mask_slice,
    f'(c) Top-3 union (GT∈Top-3 → GT color; else red).\n'
    f'Top-3 coverage = {coverage:.1f}% (missed = {missed:.1f}%)',
    show_red_mask=True, red_mask_data=red_mask
)

# ---------- (d) Uncertainty: Margin + Entropy ----------
# 若已有 margin / entropy_map，可直接替换计算段
p_sorted = np.sort(p_pred_slice, axis=-1)
p1, p2 = p_sorted[..., -1], p_sorted[..., -2]
margin_map = p1 - p2

eps = 1e-12
p_safe = np.clip(p_pred_slice, eps, 1.0)
if ENTROPY_UNIT.lower() == "bits":
    entropy_map = -np.sum(p_safe * (np.log(p_safe) / np.log(2.0)), axis=-1)
    entropy_unit_label = "bits"
else:  # nats
    entropy_map = -np.sum(p_safe * np.log(p_safe), axis=-1)
    entropy_unit_label = "nats"

roi = mask_slice > 0
if roi.any():
    margin_min, margin_max = margin_map[roi].min(), margin_map[roi].max()
    margin_norm = (margin_map - margin_min) / (margin_max - margin_min + 1e-8)
    margin_norm[~roi] = 0.0
    ent_roi = entropy_map[roi]
    ent_vmax = np.percentile(ent_roi, 95)
    hi_thr = np.percentile(ent_roi, 75)
else:
    margin_norm = np.zeros_like(margin_map)
    ent_vmax = entropy_map.max() if np.isfinite(entropy_map.max()) else 1.0
    hi_thr = 0.0

axes[3].imshow(margin_norm, cmap='gray', aspect='auto', interpolation='bilinear', vmin=0, vmax=1)
entropy_masked = np.ma.masked_where(~roi, entropy_map)
im = axes[3].imshow(entropy_masked, cmap='hot', aspect='auto', interpolation='bilinear',
                    alpha=0.70, vmin=0.0, vmax=ent_vmax)

axes[3].contour(mask_slice, levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--', alpha=0.5)
if roi.any():
    axes[3].contour(entropy_map, levels=[hi_thr], colors='yellow', linewidths=2.0, alpha=0.85)

def panel_title(ax, s):
    """面板内标题：逐段换行，保留手动换行与空行"""
    lines = s.split('\n')  # 手动分段
    wrapped = [textwrap.fill(line, width=24) if line.strip() != '' else '' for line in lines]
    s2 = '\n'.join(wrapped)  # 保留空行 => 两个 \n 连在一起
    ax.text(
        0.01, 0.99, s2,
        transform=ax.transAxes, va='top', ha='left',
        fontsize=13, fontweight='bold', color='white',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=(0,0,0,0.55), edgecolor='none'),
        multialignment='left'
    )


panel_title(
    axes[3],
    '(d) Uncertainty\n'
    'Background = Margin (p1 - p2)\n'
    f'Overlay = Entropy H(p) [{entropy_unit_label}]'
)

axes[3].axis('off')

# ---------- 右侧图例 ----------
legend_handles = []
for class_idx in [0, 2, 7, 10, 41, 42]:  # 示例：可根据需要调整
    if class_idx in LABEL_NAMES:
        legend_handles.append(
            Patch(facecolor=LABEL_COLORS_RGB[class_idx], edgecolor='white',
                  label=LABEL_NAMES[class_idx])
        )
legend_handles.append(
    Patch(facecolor=RED_MASK_COLOR[:3], alpha=RED_MASK_COLOR[3],
          edgecolor='white', label='Uncovered (Red Mask)')
)

# ---------- 布局留白 + 右侧全局色条 ----------
fig.subplots_adjust(left=0.02, right=0.90, top=0.96, bottom=0.03, wspace=0.03)

cax = fig.add_axes([0.92, 0.15, 0.015, 0.70])  # [left, bottom, width, height] in figure coords
cbar = plt.colorbar(im, cax=cax)
cbar.set_label(f'Entropy ({entropy_unit_label})', rotation=270, labelpad=14, color='white', fontsize=10)
cbar.ax.tick_params(colors='white', labelsize=9)

fig.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(0.975, 0.5),
    frameon=True, facecolor='black', edgecolor='white',
    fontsize=10, labelcolor='white'
)

plt.show()
